In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv("../data/train.csv")
data.head(10)

**Alignment warning:** The filename change in the next cell is only a guess. Its durations describe candidate recordings; it does not establish that those recordings say the CSV text. Do not use these pairs for ASR training until the source manifest or manual review confirms them.

In [ ]:
import re
import subprocess
import wave
from pathlib import Path

recordings_dir = Path("../data/DATASET")

def get_duration(filename):

    recording_name = filename.replace("sentence_kaa0", "sentence_kaa1", 1)
    recording_path = recordings_dir / recording_name
    if not recording_path.is_file():
        return float("nan")

    with recording_path.open("rb") as audio_file:
        header = audio_file.read(12)

    if header[:4] == b"RIFF" and header[8:12] == b"WAVE":
        with wave.open(str(recording_path), "rb") as recording:
            return recording.getnframes() / recording.getframerate()


    if header[4:8] == b"ftyp":
        result = subprocess.run(
            ["afinfo", str(recording_path)], capture_output=True, text=True, check=True
        )
        match = re.search(r"estimated duration:\s*([\d.]+)\s*sec", result.stdout)
        if match:
            return float(match.group(1))
        raise ValueError(f"Could not read audio duration from {recording_path}")

    raise ValueError(f"Unsupported audio format: {recording_path}")


data["duration"] = data["filename"].map(get_duration)

In [ ]:
len(data.iloc[1,0])

In [ ]:
data.describe()

In [ ]:
data["duration"].describe()

In [ ]:
data.head(20)

In [ ]:
number = 6

text = data.iloc[(number-1),1]

print(f"number: {number}")
print(f"\ntext: {text}")

In [ ]:
texts = data.loc[:, "text"].str.startswith("aqsana", na=False)
texts.value_counts()

## Check audio and transcript alignment

An exact filename match is the first check. The proposed `kaa0` to `kaa1` substitution is shown separately because it has no verified mapping. Repeated text is a review signal, not proof that a recording is wrong.

In [ ]:
from collections import Counter

audio_paths = {path.name: path for path in recordings_dir.glob("sentence_kaa*.wav")}
csv_names = set(data["filename"])
text_counts = Counter(data["text"])
candidate_names = data["filename"].str.replace("sentence_kaa0", "sentence_kaa1", n=1, regex=False)
print(f"CSV rows: {len(data):,}; audio files: {len(audio_paths):,}")
print(f"Exact filename matches: {len(csv_names & audio_paths.keys()):,}")
print(f"Unverified substitute-name matches: {candidate_names.isin(audio_paths).sum():,}")
print(f"Distinct texts: {len(text_counts):,}; texts repeated: {sum(n > 1 for n in text_counts.values()):,}")
audio_formats = Counter()
for path in audio_paths.values():
    with path.open("rb") as audio_file:
        header = audio_file.read(12)
    if header[:4] == b"RIFF" and header[8:12] == b"WAVE":
        with wave.open(str(path), "rb") as recording:
            audio_formats[f"WAV {recording.getframerate()} Hz, {recording.getnchannels()} channel(s)"] += 1
    elif header[4:8] == b"ftyp":
        audio_formats["MP4 container with .wav extension"] += 1
    else:
        audio_formats["unknown container"] += 1
print(f"Audio formats: {dict(audio_formats)}")